In [ ]:
!pip install --upgrade pip
!pip install torch transformers datasets accelerate peft bitsandbytes
!pip install wandb tensorboard seqeval scikit-learn
!pip install sentencepiece protobuf
!!pip install jsonlines
!pip install -U unsloth trl

In [ ]:
! pip install lm_eval  langdetect -q
! pip install git+https://github.com/felipemaiapolo/tinyBenchmarks

In [ ]:
!huggingface-cli login

In [ ]:
from lm_eval import evaluator
from joblib import dump



results = evaluator.simple_evaluate(
    model="hf",
    model_args="pretrained=Qwen/Qwen2.5-1.5B-Instruct,parallelize=True,trust_remote_code=True",
    tasks=["hellaswag"],
    device="cuda",
    batch_size="auto"
)

print(results)
dump(results, "results.joblib")

In [ ]:
from datasets import load_dataset
import json
import random
from pathlib import Path

# -----------------------------
# Config
# -----------------------------
DATASET_NAME = "ai4privacy/pii-masking-200k"
OUTPUT_DIR = Path("./normalized_pii_dataset")
OUTPUT_DIR.mkdir(exist_ok=True)

SEED = 42
TRAIN_RATIO = 0.7
DEV_RATIO = 0.2
VAL_RATIO = 0.1

random.seed(SEED)

# -----------------------------
# Entity Mapping
# -----------------------------
PII_TO_ENTITY = {
    "EMAIL": "email-address",
    "USERNAME": "username",
    "PASSWORD": "password",
    "ACCOUNTNAME": "username",
    "ACCOUNTNUMBER": "us-bank-account-number",

    "FIRSTNAME": "person-name",
    "MIDDLENAME": "person-name",
    "LASTNAME": "person-name",
    "PREFIX": "person-name",


    "PHONENUMBER": "phone-number",
    "PHONEIMEI": "phone-number",

    "CREDITCARDNUMBER": "credit-card-number",

    "IBAN": "iban-code",
    "BIC": "swift-code",

    "SSN": "us-ssn",


    "IP": "ip-address",
    "IPV4": "ip-address",
    "IPV6": "ip-address",
    "URL": "url",

    "DOB": "date-of-birth",

    "VEHICLEVIN": "vehicle-vin",
    "VEHICLEVRM": "vehicle-vin",
}

# -----------------------------
# Load Dataset
# -----------------------------
dataset = load_dataset(DATASET_NAME, split="train")

processed = []

for item in dataset:
    entities = []
    for span in item["privacy_mask"]:
        src_label = span["label"]
        if src_label not in PII_TO_ENTITY:
            continue

        entities.append({
            "entity_type": PII_TO_ENTITY[src_label],
            "start": span["start"],
            "end": span["end"],
            "text": item["source_text"][span["start"]:span["end"]]
        })

    if entities:
        processed.append({
            "text": item["source_text"],
            "entities": entities
        })

# -----------------------------
# Shuffle & Split
# -----------------------------
random.shuffle(processed)

n = len(processed)
train_end = int(n * TRAIN_RATIO)
dev_end = train_end + int(n * DEV_RATIO)

train_data = processed[:train_end]
dev_data = processed[train_end:dev_end]
val_data = processed[dev_end:]

# -----------------------------
# Save JSONL
# -----------------------------
def save_jsonl(data, path):
    with open(path, "w", encoding="utf-8") as f:
        for row in data:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

save_jsonl(train_data, OUTPUT_DIR / "train.jsonl")
save_jsonl(dev_data, OUTPUT_DIR / "dev.jsonl")
save_jsonl(val_data, OUTPUT_DIR / "val.jsonl")

print("✅ Done!")
print(f"Train: {len(train_data)}")
print(f"Dev:   {len(dev_data)}")
print(f"Val:   {len(val_data)}")


In [ ]:
PROMPT = """

You are a precise information extraction system.

Your task is to extract personally identifiable information (PII) entities from the given text.

ONLY extract entities belonging to the following types:
- email-address
- username
- password
- us-bank-account-number
- person-name
- phone-number
- credit-card-number
- iban-code
- swift-code
- us-ssn
- ip-address
- url
- date-of-birth
- vehicle-vin

Rules:
1. Return ONLY valid JSON. Do NOT include explanations or extra text.
2. Use the EXACT entity type names listed above.
3. Extract the entity spans exactly as they appear in the text.
4. Use character-level offsets based on the original text.
5. The "text" field MUST match the substring from start to end.
6. If no entities are found, return: {"entities": []}
7. Do NOT guess or infer missing information.

Output format:
{
  "entities": [
    {
      "entity_type": "<one of the allowed types>",
      "text": "<exact substring>"
    }
  ]
}

Make sure you only output valid json

Text:

"""

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen2.5-1.5B-Instruct",
    max_seq_length=2048,
    dtype=torch.float16,
    load_in_4bit=True
)

In [ ]:
def run_inference(text):
    prompt = PROMPT + text
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=128,
            temperature=0.0,
            do_sample=False
        )

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    decoded = extract_entities_from_output(decoded)
    print("decoded", decoded)
    match = re.search(r"\{[\s\S]*\}", decoded)
    if not match:
        return []

    try:
        parsed = json.loads(match.group())
        return parsed.get("entities", [])
    except Exception:
        return []

In [ ]:
def build_prompt(text):
    return PROMPT + text

In [ ]:
def parse_output(output):
    match = re.search(r"\{[\s\S]*\}", output)
    if not match:
        return []
    try:
        return json.loads(match.group()).get("entities", [])
    except:
        return []

In [ ]:
import json

def extract_entities_from_output(text):
    """
    Safely extracts the FIRST valid JSON object from LLM output.
    Works even if extra junk appears after it.
    """

    start = text.find("{")
    if start == -1:
        return []

    brace_count = 0
    for i in range(start, len(text)):
        if text[i] == "{":
            brace_count += 1
        elif text[i] == "}":
            brace_count -= 1

        if brace_count == 0:
            candidate = text[start:i+1]
            try:
                parsed = json.loads(candidate)
                return parsed.get("entities", [])
            except:
                return []

    return []


In [ ]:
def normalize_entities(entities):
    return {
        (e["entity_type"].strip(), e["text"].strip())
        for e in entities
        if "entity_type" in e and "text" in e
    }

In [ ]:
def batch_inference(texts, batch_size=64):
    all_predictions = []

    for i in tqdm(range(0, len(texts), batch_size)):
        batch = texts[i:i + batch_size]
        prompts = [build_prompt(t) for t in batch]
        print(prompts)
        inputs = tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True
        ).to("cuda")

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=128,
                temperature=0.0,
                do_sample=False
            )

        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        print(decoded)
        for out in decoded:
            all_predictions.append(parse_output(out))

    return all_predictions

In [ ]:
# -------------------------------
# Evaluation with DataFrames
# -------------------------------
from tqdm import tqdm
import re
import pandas as pd
def evaluate_jsonl(path, batch_size=64):
    texts = []
    gold_entities = []

    # Load JSONL
    cunt = 0
    with open(path, "r") as f:
        for line in f:
            cunt += 1
            row = json.loads(line)
            texts.append(row["text"])
            gold_entities.append(normalize_entities(row["entities"]))
            if cunt == 1000:
              break

    # Run model
    preds_raw = batch_inference(texts, batch_size)
    preds = [normalize_entities(p) for p in preds_raw]
    print(preds)
    rows = []
    entity_stats = {}

    total_tp = total_fp = total_fn = 0

    for i, (text, gold, pred) in enumerate(zip(texts, gold_entities, preds)):
        tp = gold & pred
        fp = pred - gold
        fn = gold - pred

        total_tp += len(tp)
        total_fp += len(fp)
        total_fn += len(fn)

        rows.append({
            "doc_id": i,
            "text": text,
            "gold_entities": list(gold),
            "predicted_entities": list(pred),
            "TP": len(tp),
            "FP": len(fp),
            "FN": len(fn)
        })

        for e in tp:
            entity_stats.setdefault(e[0], {"tp": 0, "fp": 0, "fn": 0})
            entity_stats[e[0]]["tp"] += 1

        for e in fp:
            entity_stats.setdefault(e[0], {"tp": 0, "fp": 0, "fn": 0})
            entity_stats[e[0]]["fp"] += 1

        for e in fn:
            entity_stats.setdefault(e[0], {"tp": 0, "fp": 0, "fn": 0})
            entity_stats[e[0]]["fn"] += 1

    # -------- DataFrames --------
    df_docs = pd.DataFrame(rows)

    entity_rows = []
    for ent, v in entity_stats.items():
        p = v["tp"] / (v["tp"] + v["fp"] + 1e-9)
        r = v["tp"] / (v["tp"] + v["fn"] + 1e-9)
        f1 = 2 * p * r / (p + r + 1e-9)

        entity_rows.append({
            "entity_type": ent,
            "tp": v["tp"],
            "fp": v["fp"],
            "fn": v["fn"],
            "precision": round(p, 4),
            "recall": round(r, 4),
            "f1": round(f1, 4)
        })

    df_entity = pd.DataFrame(entity_rows)

    overall = {
        "precision": total_tp / (total_tp + total_fp + 1e-9),
        "recall": total_tp / (total_tp + total_fn + 1e-9),
        "f1": 2 * (total_tp / (total_tp + total_fp + 1e-9)) *
              (total_tp / (total_tp + total_fn + 1e-9)) /
              ((total_tp / (total_tp + total_fp + 1e-9)) +
               (total_tp / (total_tp + total_fn + 1e-9)) + 1e-9)
    }

    df_overall = pd.DataFrame([overall])

    return df_docs, df_entity, df_overall

In [ ]:
evaluate_jsonl('/content/normalized_pii_dataset/val.jsonl')

In [ ]:
def evaluate(dataset_path, batch_size=8):
    with open(dataset_path) as f:
        data = json.load(f)

    texts = [row["text"] for row in data]
    golds = [normalize_entities(row["entities"]) for row in data]

    preds_raw = batch_inference(texts, batch_size)
    preds = [normalize_entities(p) for p in preds_raw]

    total_tp = total_fp = total_fn = 0
    per_entity = {}

    for gold, pred in zip(golds, preds):
        tp = gold & pred
        fp = pred - gold
        fn = gold - pred

        total_tp += len(tp)
        total_fp += len(fp)
        total_fn += len(fn)

        for e in tp:
            per_entity.setdefault(e[0], {"tp": 0, "fp": 0, "fn": 0})
            per_entity[e[0]]["tp"] += 1

        for e in fp:
            per_entity.setdefault(e[0], {"tp": 0, "fp": 0, "fn": 0})
            per_entity[e[0]]["fp"] += 1

        for e in fn:
            per_entity.setdefault(e[0], {"tp": 0, "fp": 0, "fn": 0})
            per_entity[e[0]]["fn"] += 1

    # ---- Metrics ----
    precision = total_tp / (total_tp + total_fp + 1e-9)
    recall = total_tp / (total_tp + total_fn + 1e-9)
    f1 = 2 * precision * recall / (precision + recall + 1e-9)

    print("\n========== OVERALL ==========")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1 Score:  {f1:.4f}")

    print("\n====== PER ENTITY ======")
    for ent, stats in per_entity.items():
        p = stats["tp"] / (stats["tp"] + stats["fp"] + 1e-9)
        r = stats["tp"] / (stats["tp"] + stats["fn"] + 1e-9)
        f1 = 2 * p * r / (p + r + 1e-9)
        print(f"{ent:25s} | P={p:.3f} | R={r:.3f} | F1={f1:.3f}")

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "per_entity": per_entity
    }


# -------------------------------
# Run
# -------------------------------
if __name__ == "__main__":
    evaluate("eval_data.json", batch_size=8)

In [ ]:
texts = []
with open("/content/normalized_pii_dataset/val.jsonl") as f:
    for line in f:
        row = json.loads(line)
        texts.append(row["text"])


In [ ]:
texts[1]

In [ ]:
op = run_inference(texts[1])
print("op---------",op)
if op:
  extract_entities_from_output(op)